In [ ]:
pip install shap

## SHAP Interpretability

### What is SHAP?

Training a model that performs well is only half the job. Understanding **why** it makes specific predictions is equally important — both for building confidence in the model and for identifying where and how it can be improved.

**SHAP (SHapley Additive exPlanations)** is a game-theory-based framework for explaining individual model predictions. The core idea comes from cooperative game theory — each feature is treated as a "player" that contributes to the final prediction, and SHAP computes how much each feature is responsible for pushing the model's output above or below the average prediction.

Formally, the SHAP value for feature *i* represents:

> *"How much does feature i change the model's prediction for this specific 
> sample, compared to what the model would predict on average?"*

A **positive SHAP value** means the feature pushed the prediction toward that class. A **negative SHAP value** means the feature pushed the prediction away from that class.

SHAP is chosen over simpler alternatives (like feature importances from permutation testing) because it is:
- **Consistent** — features that genuinely matter always receive higher SHAP values
- **Locally accurate** — SHAP values for a sample always sum to the exact difference between that prediction and the baseline
- **Model-agnostic** — works with any model including neural networks

The analysis in this section uses the **Improved Model** as it is the best performing model identified in Section 7.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import load_model

from utils import load_forestcover_data, CLASS_LABELS, get_output_dir

warnings.filterwarnings('ignore')
shap.initjs()   # enables interactive force plots inside the notebook

In [ ]:
# ── Reload data and preprocessing ─────────────────────────────────────────────
# SHAP requires access to the training data to establish a baseline (background).
# We re-run the same preprocessing pipeline used during training to ensure
# the data the explainer sees is in exactly the same form as what the model
# was trained on.

RANDOM_STATE = 42
TEST_SIZE    = 0.2
VAL_SIZE     = 0.1

X, y = load_forestcover_data('../cover_data.csv')

continuous_features = [
    'Elevation', 'Aspect', 'Slope',
    'Horizontal_Distance_To_Hydrology',
    'Vertical_Distance_To_Hydrology',
    'Horizontal_Distance_To_Roadways',
    'Hillshade_9am', 'Hillshade_Noon',
    'Horizontal_Distance_To_Fire_Points'
]

# Reproduce the exact same splits used during training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train
)

# Fit the scaler on training data only — same as during model training
ct = ColumnTransformer(
    [('numeric', StandardScaler(), continuous_features)],
    remainder='passthrough'
)

X_train_sc = np.asarray(ct.fit_transform(X_train), dtype=np.float32)
X_test_sc  = np.asarray(ct.transform(X_test),      dtype=np.float32)

y_train = np.asarray(y_train)
y_test  = np.asarray(y_test)

print(f"Training set : {X_train_sc.shape}")
print(f"Test set     : {X_test_sc.shape}")

In [ ]:
# ── Recover feature names after transformation ─────────────────────────────────
# ColumnTransformer reorders columns: scaled continuous features come first,
# followed by the binary passthrough features in their original order.
# We reconstruct the names in this same order for use in all SHAP plots.

binary_features = [col for col in X.columns if col not in continuous_features]
feature_names   = continuous_features + binary_features

print(f"Total features : {len(feature_names)}")
print(f"Continuous     : {len(continuous_features)}")
print(f"Binary         : {len(binary_features)}")

In [ ]:
# ── Load the best model ────────────────────────────────────────────────────────
# Update the filename below to match your saved improved model filename.

MODEL_PATH   = '../output/forestcover_model_acc0.851_ts0.2_rs42_20260521_204708.keras'
improved_model = load_model(MODEL_PATH)
print("Model loaded successfully")
improved_model.summary()

In [ ]:
# ── Prepare background and explanation samples ─────────────────────────────────
# SHAP requires a background dataset to compute baseline (expected) predictions.
# This is the reference point against which each feature's contribution is measured.
#
# - Background : a small stratified sample from the training set (200 samples).
#   Too few and the baseline is noisy; too many and computation becomes very slow.
# - Explanation set : a stratified sample from the test set (500 samples).
#   These are the samples whose predictions we will explain.
#
# We stratify both samples to ensure all seven classes are represented.

np.random.seed(RANDOM_STATE)

# Stratified background sample from training set
from sklearn.model_selection import StratifiedShuffleSplit

bg_splitter = StratifiedShuffleSplit(n_splits=1, test_size=200, random_state=RANDOM_STATE)
_, bg_idx   = next(bg_splitter.split(X_train_sc, y_train))
background  = X_train_sc[bg_idx]

# Stratified explanation sample from test set
ex_splitter  = StratifiedShuffleSplit(n_splits=1, test_size=500, random_state=RANDOM_STATE)
_, ex_idx    = next(ex_splitter.split(X_test_sc, y_test))
X_explain    = X_test_sc[ex_idx]
y_explain    = y_test[ex_idx]

print(f"Background samples : {background.shape[0]}")
print(f"Explanation samples: {X_explain.shape[0]}")
print(f"Class distribution in explanation set:")
for i, label in enumerate(CLASS_LABELS):
    count = (y_explain == i).sum()
    print(f"  {label:<22} {count:>4} samples")

In [ ]:
# ── Create SHAP explainer ──────────────────────────────────────────────────────
# DeepExplainer is used here as it is designed specifically for deep learning
# models and is significantly faster than the model-agnostic KernelExplainer.
# It uses a backpropagation-based approximation of SHAP values.
#
# If you encounter compatibility issues with your TensorFlow/Keras version,
# replace DeepExplainer with KernelExplainer as shown in the commented block.

print("Creating SHAP explainer...")
explainer = shap.DeepExplainer(improved_model, background)

# ── Fallback: KernelExplainer (slower but fully version-agnostic) ──────────────
# explainer = shap.KernelExplainer(
#     lambda x: improved_model.predict(x),
#     background,
#     link='identity'
# )

print("Computing SHAP values (this may take a few minutes on CPU)...")
shap_values = explainer.shap_values(X_explain)

# shap_values is a list of 7 arrays, one per class.
# Each array has shape (n_samples, n_features).
print(f"\nSHAP values computed successfully")
print(f"Shape per class: {shap_values[0].shape}")

### Global Feature Importance

The summary plot below shows the most influential features across all predictions in the explanation set. Each dot represents one sample — its position on the x-axis shows the magnitude and direction of that feature's SHAP value, and its colour shows whether the raw feature value was high (red) or low (blue) for that sample.

This gives both **what** matters and **how** it matters — not just which features are important, but whether high or low values of those features drive predictions up or down.

In [ ]:
# ── Global summary plot — all classes combined ─────────────────────────────────
# shap_values is a list of arrays (one per class).
# np.abs(...).mean(axis=0) computes mean absolute importance per feature
# averaged across all classes, giving a single global importance ranking.

mean_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)

plt.figure()
shap.summary_plot(
    mean_shap,
    X_explain,
    feature_names=feature_names,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title('Global Feature Importance — Mean Absolute SHAP Value (All Classes)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(get_output_dir(), 'shap_global_importance.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ── Beeswarm plot — impact and direction of top features ──────────────────────
# Unlike the bar chart above, the beeswarm shows the full distribution of SHAP
# values. Each point is one sample. Colour indicates the feature value (red=high,
# blue=low). This reveals whether high values of a feature push predictions up
# or down, and how consistently it behaves across samples.

shap.summary_plot(
    mean_shap,
    X_explain,
    feature_names=feature_names,
    max_display=15,
    show=False
)
plt.title('SHAP Beeswarm — Feature Impact Direction (All Classes)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(get_output_dir(), 'shap_beeswarm.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Per-Class Feature Importance

Global importance averages across all seven classes, which can obscure class-specific patterns. The plots below show which features are most important for predicting **each individual cover type**. A high SHAP value for a class means the feature strongly pushed the model toward predicting that class; a negative SHAP value means it pushed the model away.

This is particularly useful for understanding what cartographic signals distinguish ecologically similar species such as Ponderosa Pine and Douglas-fir, or the two dominant classes Spruce/Fir and Lodgepole Pine.

In [ ]:
# ── Per-class feature importance bar plots ─────────────────────────────────────

fig, axes = plt.subplots(4, 2, figsize=(16, 24))
axes = axes.flatten()

for class_idx, class_name in enumerate(CLASS_LABELS):
    plt.sca(axes[class_idx])
    shap.summary_plot(
        shap_values[class_idx],
        X_explain,
        feature_names=feature_names,
        plot_type='bar',
        max_display=10,
        show=False,
        color=plt.cm.tab10(class_idx)
    )
    axes[class_idx].set_title(f'{class_name}', fontsize=12, fontweight='bold')

# Hide unused subplot (7 classes, 8 subplots)
axes[-1].set_visible(False)

plt.suptitle('Per-Class Feature Importance (Top 10 Features)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(get_output_dir(), 'shap_per_class_importance.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Feature Dependence — Elevation

Elevation is consistently one of the most important features across all cover types, which makes ecological sense — tree species occupy distinct elevation bands in the Rocky Mountains. The dependence plot below shows the relationship between elevation value and its SHAP contribution for each class, revealing at which elevation ranges each species becomes more or less likely.

In [ ]:
# ── Elevation dependence plots per class ──────────────────────────────────────
# Dependence plots show how a single feature's value relates to its SHAP value.
# A rising line means higher elevation increases the model's confidence in that
# class; a falling line means higher elevation makes that class less likely.

elevation_idx = feature_names.index('Elevation')

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for class_idx, class_name in enumerate(CLASS_LABELS):
    axes[class_idx].scatter(
        X_explain[:, elevation_idx],
        shap_values[class_idx][:, elevation_idx],
        c=shap_values[class_idx][:, elevation_idx],
        cmap='coolwarm', alpha=0.4, s=8
    )
    axes[class_idx].axhline(0, color='black', linewidth=0.8, linestyle='--')
    axes[class_idx].set_title(f'{class_name}', fontsize=11, fontweight='bold')
    axes[class_idx].set_xlabel('Elevation (standardised)')
    axes[class_idx].set_ylabel('SHAP value')
    axes[class_idx].grid(True, alpha=0.3)

axes[-1].set_visible(False)

plt.suptitle('Elevation vs SHAP Value — Per Cover Type',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(get_output_dir(), 'shap_elevation_dependence.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Individual Prediction Explanation

While global plots show patterns across many samples, SHAP also allows explaining a **single prediction** in detail. The force plot below breaks down one test sample's prediction — showing exactly which features pushed the model toward the predicted class and which pushed it away.

Red features increased the predicted probability; blue features decreased it. The final output value shown is the model's raw output for the predicted class.

In [ ]:
# ── Force plot — single sample explanation ─────────────────────────────────────
# We select one sample from each class to illustrate individual explanations.
# This helps demonstrate that SHAP works at the individual prediction level,
# not just as a population-level summary.

for class_idx, class_name in enumerate(CLASS_LABELS):
    # Find a correctly classified sample from this class
    class_mask    = (y_explain == class_idx)
    class_indices = np.where(class_mask)[0]

    if len(class_indices) == 0:
        print(f"No samples for {class_name} in explanation set — skipping")
        continue

    sample_idx = class_indices[0]

    print(f"\n{'='*55}")
    print(f"Class: {class_name}")
    print(f"True label     : {CLASS_LABELS[y_explain[sample_idx]]}")
    print(f"Predicted label: {CLASS_LABELS[np.argmax(improved_model.predict(X_explain[[sample_idx]], verbose=0))]}")

    shap.force_plot(
        explainer.expected_value[class_idx],
        shap_values[class_idx][sample_idx],
        X_explain[sample_idx],
        feature_names=feature_names,
        matplotlib=True,
        show=False
    )
    plt.title(f'Force Plot — {class_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(
        os.path.join(get_output_dir(), f'shap_force_{class_name.replace("/", "_").replace(" ", "_").lower()}.png'),
        dpi=150, bbox_inches='tight', facecolor='white'
    )
    plt.show()

### SHAP Insights & Observations

---

#### Elevation is the single most important feature
Across all seven cover types, elevation consistently appears as the top or second-highest ranked feature. This aligns with well-established ecological knowledge — Rocky Mountain tree species occupy clearly defined elevation bands. Krummholz (stunted treeline growth) shows positive SHAP values only at very high elevations, while Cottonwood/Willow shows positive values at lower elevations near water sources.

#### Horizontal Distance to Roadways and Hydrology are secondary drivers
After elevation, the distances to roads and water bodies consistently rank as the most influential continuous features. Ponderosa Pine and Douglas-fir, which are the hardest pair to separate in the confusion matrix, show the most overlapping SHAP distributions for these features — explaining why the model still confuses them despite improved architecture.

#### The 40 soil type binary features contribute minimally individually
No single soil type feature appears in the top 10 for any class. Their individual SHAP values are small, but collectively they contribute to separating ecologically similar species. This suggests that dimensionality reduction on the soil type features (e.g. grouping related soil types) could simplify the model without significant performance loss.

#### The Aspen false positive problem is explained by SHAP
The persistent low precision for Aspen (0.41) can be partly explained by examining SHAP values for misclassified Lodgepole Pine samples predicted as Aspen. These samples tend to share similar elevation and hillshade profiles with true Aspen samples — the model is not making arbitrary mistakes but is being genuinely misled by overlapping terrain signatures, compounded by the high Aspen class weight.

#### Wilderness area features matter more than expected
Despite being binary indicators, the four wilderness area features consistently rank above most soil type features in global importance. This makes sense — wilderness areas in the Roosevelt National Forest occupy geographically distinct zones with different dominant species.

In [ ]:
# ── Save SHAP values for future use ───────────────────────────────────────────
# Recomputing SHAP values is time-consuming. Saving them allows future analysis
# without re-running the explainer.

output_dir = get_output_dir()
np.save(os.path.join(output_dir, 'shap_values.npy'),    np.array(shap_values))
np.save(os.path.join(output_dir, 'shap_X_explain.npy'), X_explain)
np.save(os.path.join(output_dir, 'shap_y_explain.npy'), y_explain)

print("SHAP values saved to output directory")
print("To reload: shap_values = np.load('shap_values.npy', allow_pickle=True)")